# Week 5 — Agents, tools, MCP, state, and approval

Compare in-process, prompt, Hosted, and self-hosted agents. Hosting and protocol are separate choices. Give every tool a narrow schema and deterministic authorization; the model's request is never authorization.

In [ ]:
import importlib.util
import sys
from pathlib import Path

curriculum_root = next(
    candidate
    for base in (Path.cwd(), *Path.cwd().parents)
    for candidate in (base, base / "examples" / "foundry-curriculum")
    if (candidate / "notebook_setup.py").is_file()
)
spec = importlib.util.spec_from_file_location(
    "foundry_curriculum_setup", curriculum_root / "notebook_setup.py"
)
helpers = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = helpers
spec.loader.exec_module(helpers)
session = helpers.load_session(curriculum_root)
labs = helpers.load_offline_labs(curriculum_root)
session.safe_summary()

In [ ]:
TOOL_POLICY = {
    "search_curriculum": labs.ToolPolicy(
        frozenset({"foundry-learners"}), side_effect=False, max_attempts=1
    ),
    "publish_release": labs.ToolPolicy(
        frozenset({"release-owners"}), side_effect=True, max_attempts=2
    ),
}
tool_backend = labs.SimulatedToolBackend({"publish_release": 1})
tool_gateway = labs.ToolGateway(TOOL_POLICY, tool_backend)
TOOL_POLICY

In [ ]:
allowed_read = tool_gateway.execute(
    "search_curriculum",
    {"query": "Foundry endpoint"},
    caller_groups=frozenset({"foundry-learners"}),
    approved=False,
    idempotency_key="read-001",
)
denied_read = tool_gateway.execute(
    "search_curriculum",
    {"query": "private"},
    caller_groups=frozenset({"untrusted"}),
    approved=False,
    idempotency_key="read-002",
)

In [ ]:
approval_block = tool_gateway.execute(
    "publish_release",
    {"version": "v2"},
    caller_groups=frozenset({"release-owners"}),
    approved=False,
    idempotency_key="release-042",
)
published = tool_gateway.execute(
    "publish_release",
    {"version": "v2"},
    caller_groups=frozenset({"release-owners"}),
    approved=True,
    idempotency_key="release-042",
)
replayed = tool_gateway.execute(
    "publish_release",
    {"version": "v2"},
    caller_groups=frozenset({"release-owners"}),
    approved=True,
    idempotency_key="release-042",
)

In [ ]:
idempotency_conflict = tool_gateway.execute(
    "publish_release",
    {"version": "v3"},
    caller_groups=frozenset({"release-owners"}),
    approved=True,
    idempotency_key="release-042",
)
assert allowed_read.status == "succeeded"
assert denied_read.status == approval_block.status == "blocked"
assert published.status == "succeeded" and published.attempts == 2
assert replayed.replayed and replayed.value == published.value
assert tool_backend.call_counts["publish_release"] == 2
assert tool_backend.committed_keys == {"release-042"}
assert idempotency_conflict.reason == "idempotency key conflict"
tool_evidence = {
    "allowed_read": allowed_read,
    "denied_read": denied_read,
    "approval_block": approval_block,
    "bounded_retry": published,
    "idempotent_replay": replayed,
}
tool_evidence

## Agent threat model

Document goal hijacking, tool misuse, identity abuse, MCP supply-chain and data-egress risk, memory poisoning, cascading failure, unbounded consumption, and human over-trust. Add step/token/time budgets, endpoint allow-lists, idempotency keys, a memory retention policy, and an interrupt before side effects.

## Exit criteria

Demonstrate an allowed read, a denied unauthorized call, a blocked side effect awaiting approval, and an idempotent retry. Treat multi-agent orchestration as an advanced core lab; keep A2A-specific work optional when its required components are preview.